<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L05-retrieval-metrics-and-pooling-bias/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L05-retrieval-metrics-and-pooling-bias/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T10-L05-retrieval-metrics-and-pooling-bias/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T10-L05-retrieval-metrics-and-pooling-bias/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T10-L05 · How good is your retriever? Recall, MRR and nDCG from scratch, the pooling trap, and synthetic labels with a known error

**You will build:** the measuring instrument that retrieval work stands on: recall@k, reciprocal
rank and nDCG@k over graded judgements, a judgement pool built from the top results of some
systems, a scripted labeller whose mistakes you planted, and a rank correlation that says how far
those two sources of error move a leaderboard.

**Time:** ~50 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T06-L01-retrieval-from-scratch (BM25 and a TF-IDF plus SVD retriever; you met
the three metrics there against a fixed answer key) and T06-L03-reranking-measured (what a first
stage and a ranked list are).

Here the answer key is the thing under test. The corpus is generated, so the true relevance of
every document is known. That lets you do what real evaluation never can: build the judgements
the way a real team does (from what some systems retrieved, or from a labeller that makes
mistakes) and then compare them with the truth.

By the end you will be able to:

1. Implement recall@k, reciprocal rank and nDCG@k over graded judgements from their
   definitions, with a stated behaviour for a query that has no relevant document.
2. Implement judgement pooling, and measure how far pooled scores sit from the planted truth
   for systems that contributed to the pool and for systems that did not.
3. Implement Kendall's tau between two orderings of the same systems, and use it to measure how
   far a pool moves a leaderboard, by pool depth and by which systems fed the pool.
4. Implement a scripted labeller with planted miss and false-alarm rates, and measure its
   realised rates against the planted ones.
5. Measure how far three kinds of planted label error move the order of the systems over repeated
   labelling seeds, and compare that movement with the movement caused by pooling.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T06-L01`, `T06-L03`

**This lesson assumes you can already:**

- write Python functions, loops and dictionaries, and index and slice numpy arrays
- read a ranked list of search results and say what 'the top 10' of it is
- recognise recall, precision and nDCG by name (T06-L01 builds them against a fixed answer key)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Introduction to Information Retrieval, chapter 8, Evaluation in information retrieval](https://nlp.stanford.edu/IR-book/html/htmledition/evaluation-in-information-retrieval-1.html) · Manning, Raghavan and Schütze (Cambridge University Press) · *free* — the whole book is free online; chapter 8 covers test collections, relevance judgements, pooling, nDCG and assessor agreement
- [Bias and the Limits of Pooling for Large Collections](https://www.nist.gov/publications/bias-and-limits-pooling-large-collections) · Buckley, Dimmick, Soboroff and Voorhees (Information Retrieval, 2007; NIST hosts the full paper) · *free* — the research behind this lesson's pooling sections, on how a pool of too few documents biases a test collection
- [Large language models can accurately predict searcher preferences](https://arxiv.org/abs/2309.10621) · Thomas, Spielman, Craswell and Mitra (arXiv 2309.10621) · *free* — a worked account of using a model as the labeller and measuring it against trusted labels, the setting of sections 7 and 8
- [NumPy, the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) · The NumPy project · *free* — arrays, indexing and averaging, the few numpy operations the scoring code leans on
- [The Python Tutorial, sections 4 and 5 (more control flow tools; data structures)](https://docs.python.org/3/tutorial/) · The Python Software Foundation · *free* — the official tutorial; defining functions, for loops, lists, sets and dictionaries are the plain Python every exercise here is written in

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import itertools
import math
import os
import re
import signal
import sys
import threading
import time
import traceback
from typing import Callable, Mapping, NamedTuple, Sequence

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20261002          # this lesson's one fixed seed: the corpus, the queries and every label
                          # error derive from it, so re-running prints the same numbers anywhere.
EVAL_K = 10              # the cutoff for recall@k and nDCG@k throughout this lesson
POOL_DEPTH = 10          # how many top results of each pooled system a judge looks at
RUN_DEPTH = 100          # how many results each system returns per query (at most)

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("recall_at_k", "reciprocal_rank"),
    "exercise 2": ("ndcg_at_k",),
    "exercise 3": ("pooled_qrels",),
    "exercise 4": ("kendall_tau",),
    "exercise 5": ("apply_labeller",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (pooled_qrels)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A check that never returns would freeze the notebook (and the grader, which imports it). Every
# check below runs under a time limit far above what a correct answer needs, and so does every
# single fit a demo asks your code for; a whole demo gets a much longer limit, because slow but
# correct code is allowed to be slow.
_CHECK_SECONDS = 2
_CALL_SECONDS = 10
_DEMO_SECONDS = 10


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that whoever started us had already set
    keeps its own deadline if it is due first, and is put back, with the time it had left and
    its repeat interval, when the block ends. A correct answer finishes far inside the limit,
    and nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose "
               "stopping condition is never met (a counter that is never "
               "advanced, an index that wraps round, a tolerance that is never "
               "reached exactly) runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. A test collection with the truth planted in it

A **test collection** is three things: documents, queries, and **judgements** (often called
qrels): for some query-document pairs, how relevant the document is. Everything an evaluation
reports is computed from those three. If the judgements are wrong or incomplete, every number
downstream is, and no amount of care in the metric code can notice.

The collection below is generated, so the truth is known. Each topic has a few **head words**
(what a searcher would type) and more **tail words** (the vocabulary that surrounds the topic).
These kinds of document are planted:

* **direct**: head words and tail words together. True grade 2.
* **mismatch**: tail words only. It is about the topic and has none of the head words of its own, so
  a word-matching system can almost never see it (a stray word, below, occasionally gives it
  one). True grade 1.
* **decoy**: a head word or two with unrelated content. True grade 0.
* **background**: filler text. True grade 0.

Some documents also carry a stray word or two drawn at random from the words of every topic, now
and then one of their own topic's. `TRUTH[qid][doc]` is the planted grade
of every relevant document; a document missing from it has grade 0. Real collections never have
this table: it is what every real evaluation is trying to approximate.

Read the cell below; nothing in it is graded.

In [ ]:
N_TOPICS = 12
N_HEAD, N_TAIL = 4, 8                       # head words and tail words per topic
N_DIRECT, N_MISMATCH, N_DECOY = 5, 4, 5     # documents of each kind, per topic
N_BACKGROUND = 150
QUERIES_PER_TOPIC = 5
BLEED = 0.4                                 # chance a document also carries stray words
GRADE_OF_KIND = {"direct": 2, "mismatch": 1, "decoy": 0, "background": 0}

_SYLLABLES = ["ba", "ko", "ri", "tu", "ne", "sa", "vo", "li", "mu", "de", "fa", "gi", "ha", "jo",
              "ku", "lo", "ma", "ni", "pe", "qua", "ro", "su", "ti", "va", "we", "xo", "yu", "ze"]
FILLER = ["report", "today", "please", "status", "system", "regarding", "update", "note", "team",
          "review", "notice", "summary", "item", "record", "entry", "message", "forward",
          "attached", "thanks", "daily"]


class Query(NamedTuple):
    qid: int
    topic: int
    tokens: list[str]


def _invented_words(n: int, rng, taken: set) -> list[str]:
    """`n` new three-syllable words, none of them used before (and none a filler word)."""
    out: list[str] = []
    while len(out) < n:
        word = "".join(rng.choice(_SYLLABLES, size=3).tolist())
        if word not in taken and word not in FILLER:
            taken.add(word)
            out.append(word)
    return out


def build_collection(seed: int = SEED):
    """Documents, their planted kind and topic, the queries, and the true grades. Given."""
    rng = np.random.default_rng(seed)
    taken: set = set()
    heads, tails = [], []
    for _ in range(N_TOPICS):
        heads.append(_invented_words(N_HEAD, rng, taken))
        tails.append(_invented_words(N_TAIL, rng, taken))
    every_word = [w for t in range(N_TOPICS) for w in heads[t] + tails[t]]
    docs: list[str] = []
    kinds: list[str] = []
    topics: list[int] = []

    def pick(pool, n):
        return rng.choice(pool, size=int(n), replace=False).tolist()

    def add(words, kind, topic):
        words = list(words)
        if rng.random() < BLEED:
            words += pick(every_word, rng.integers(1, 3))
        rng.shuffle(words)
        docs.append(" ".join(words))
        kinds.append(kind)
        topics.append(topic)

    for t in range(N_TOPICS):
        for _ in range(N_DIRECT):
            add(pick(heads[t], rng.integers(2, 4)) + pick(tails[t], rng.integers(4, 6))
                + pick(FILLER, 4), "direct", t)
        for _ in range(N_MISMATCH):
            add(pick(tails[t], rng.integers(6, 8)) + pick(FILLER, 3), "mismatch", t)
        for _ in range(N_DECOY):
            elsewhere = int(rng.choice([u for u in range(N_TOPICS) if u != t]))
            add(pick(heads[t], rng.integers(1, 3)) + pick(tails[elsewhere], 2)
                + pick(FILLER, 6), "decoy", t)
    for _ in range(N_BACKGROUND):
        add(rng.choice(FILLER, size=int(rng.integers(8, 13)), replace=True).tolist(),
            "background", -1)
    order = rng.permutation(len(docs))          # so a document's id says nothing about its kind
    docs = [docs[i] for i in order]
    kinds = [kinds[i] for i in order]
    topics = [topics[i] for i in order]

    qrng = np.random.default_rng(seed + 1)
    combos = [c for r in (2, 3) for c in itertools.combinations(range(N_HEAD), r)]
    queries: list[Query] = []
    truth: dict[int, dict[int, int]] = {}
    for t in range(N_TOPICS):
        for ci in sorted(qrng.choice(len(combos), size=QUERIES_PER_TOPIC, replace=False).tolist()):
            qid = len(queries)
            queries.append(Query(qid, t, [heads[t][i] for i in combos[ci]]))
            truth[qid] = {d: GRADE_OF_KIND[kinds[d]] for d in range(len(docs))
                          if topics[d] == t and GRADE_OF_KIND[kinds[d]] > 0}
    return docs, kinds, topics, queries, truth


DOCS, KINDS, TOPICS, QUERIES, TRUTH = build_collection()
print(f"{len(DOCS)} documents, {len(QUERIES)} queries, {N_TOPICS} topics")
for _kind in GRADE_OF_KIND:
    print(f"  {_kind:<10} {KINDS.count(_kind):>4} documents, planted grade {GRADE_OF_KIND[_kind]}")
_per_query = [len(TRUTH[q.qid]) for q in QUERIES]
print(f"relevant documents per query: {min(_per_query)} to {max(_per_query)} "
      f"(mean {np.mean(_per_query):.1f}); the evaluation cutoff is EVAL_K = {EVAL_K}")

_q = QUERIES[0]
print(f"\nquery {_q.qid} is {' '.join(_q.tokens)!r}; some of its relevant documents:")
for _kind in ("direct", "mismatch"):
    _doc = next(d for d in TRUTH[_q.qid] if KINDS[d] == _kind)
    print(f"  {_kind:<9} grade {TRUTH[_q.qid][_doc]}: {DOCS[_doc]}")
_pairs = [(q.qid, d) for q in QUERIES for d, g in TRUTH[q.qid].items() if g == 1]
_visible = sum(any(w in DOCS[d].split() for w in QUERIES[qid].tokens) for qid, d in _pairs)
print(f"of {len(_pairs)} (query, grade-1 document) pairs, {_visible} share a word with the query")

## 2. The systems under test

A leaderboard needs several systems. These are small versions of the retrievers from
T06-L01, copied here so this notebook runs alone: BM25 in several settings, cosine similarity in
TF-IDF space, a dense retriever (TF-IDF folded through a truncated SVD, at several sizes of latent
space) and a hybrid that fuses BM25 and the middle dense retriever by reciprocal rank fusion.
A dense one can see that a mismatch document belongs to the topic, because the tail words
co-occur with the head words in the direct documents, and how well it does so depends on the size
of its latent space (the truth leaderboard below shows it); the word-matching ones cannot.

Two details keep every printed number the same on every machine: cosines are rounded to six
decimals before they are ranked, and ties break by document id.

`RUNS[system][qid]` is that system's ranked list of document ids for the query, at most
`RUN_DEPTH` long. Read the cell below; nothing in it is graded.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")


def tokenize(text: str) -> list[str]:
    return _TOKEN_RE.findall(text.lower())


class Index(NamedTuple):
    doc_term_counts: list[dict[str, int]]
    doc_lengths: list[int]
    avgdl: float
    df: dict[str, int]
    n_docs: int


def build_index(docs: Sequence[str]) -> Index:
    """Per-document term counts and corpus statistics (T06-L01, unchanged in behaviour)."""
    counts_per_doc, lengths, df = [], [], {}
    for doc in docs:
        counts: dict[str, int] = {}
        for token in tokenize(doc):
            counts[token] = counts.get(token, 0) + 1
        counts_per_doc.append(counts)
        lengths.append(sum(counts.values()))
        for term in counts:
            df[term] = df.get(term, 0) + 1
    return Index(counts_per_doc, lengths, sum(lengths) / len(docs), df, len(docs))


def bm25_rank(query_tokens: Sequence[str], index: Index, k1: float = 1.2,
              b: float = 0.75) -> list[int]:
    """Documents with a non-zero BM25 score, best first; ties by id (T06-L01's formula)."""
    wanted: dict[str, int] = {}
    for token in query_tokens:
        wanted[token] = wanted.get(token, 0) + 1
    scored = []
    for doc_id in range(index.n_docs):
        counts, length = index.doc_term_counts[doc_id], index.doc_lengths[doc_id]
        score = 0.0
        for term, qtf in wanted.items():
            tf = counts.get(term, 0)
            if tf:
                n_t = index.df.get(term, 0)
                idf = math.log((index.n_docs - n_t + 0.5) / (n_t + 0.5))
                score += qtf * idf * tf / (k1 * ((1 - b) + b * length / index.avgdl) + tf)
        if score != 0.0:
            scored.append((doc_id, score))
    scored.sort(key=lambda pair: (-pair[1], pair[0]))
    return [doc_id for doc_id, _ in scored]


def reciprocal_rank_fusion(rankings: Sequence[Sequence[int]], k: int = 60) -> list[int]:
    """Fuse ranked lists: each document scores the sum of 1 / (k + rank) (T06-L01)."""
    totals: dict[int, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            totals[doc_id] = totals.get(doc_id, 0.0) + 1.0 / (k + rank)
    return [d for d, _ in sorted(totals.items(), key=lambda pair: (-pair[1], pair[0]))]


INDEX = build_index(DOCS)
_VOCAB = sorted(INDEX.df)
_POS = {term: i for i, term in enumerate(_VOCAB)}
_IDF = np.array([math.log(INDEX.n_docs / INDEX.df[t]) + 1.0 for t in _VOCAB])
_TFIDF = np.zeros((INDEX.n_docs, len(_VOCAB)))
for _d, _counts in enumerate(INDEX.doc_term_counts):
    for _t, _tf in _counts.items():
        _TFIDF[_d, _POS[_t]] = (1.0 + math.log(_tf)) * _IDF[_POS[_t]]
_TFIDF = _TFIDF / np.linalg.norm(_TFIDF, axis=1)[:, None]
_VT = np.linalg.svd(_TFIDF, full_matrices=False)[2]


def _query_vector(tokens: Sequence[str]) -> np.ndarray:
    counts: dict[str, int] = {}
    for token in tokens:
        counts[token] = counts.get(token, 0) + 1
    vec = np.zeros(len(_VOCAB))
    for term, tf in counts.items():
        if term in _POS:
            vec[_POS[term]] = (1.0 + math.log(tf)) * _IDF[_POS[term]]
    return vec


def _ranked_by_cosine(scores: np.ndarray, drop_zeros: bool) -> list[int]:
    scores = np.round(scores, 6)
    order = sorted(range(len(scores)), key=lambda d: (-scores[d], d))
    return [d for d in order if scores[d] > 0] if drop_zeros else order


def tfidf_rank(tokens: Sequence[str]) -> list[int]:
    vec = _query_vector(tokens)
    norm = np.linalg.norm(vec)
    return _ranked_by_cosine(_TFIDF @ (vec / norm) if norm else np.zeros(INDEX.n_docs), True)


def dense_ranker(dims: int) -> Callable[[Sequence[str]], list[int]]:
    basis = _VT[:dims]
    embedded = _TFIDF @ basis.T
    embedded = embedded / np.linalg.norm(embedded, axis=1)[:, None]

    def rank(tokens: Sequence[str]) -> list[int]:
        q = basis @ _query_vector(tokens)
        norm = np.linalg.norm(q)
        return _ranked_by_cosine(embedded @ (q / norm) if norm else np.zeros(INDEX.n_docs), False)
    return rank


_dense_small, _dense_mid, _dense_large = dense_ranker(12), dense_ranker(24), dense_ranker(48)
SYSTEMS: dict[str, Callable[[Sequence[str]], list[int]]] = {
    "bm25": lambda tokens: bm25_rank(tokens, INDEX),
    "bm25-k1-3": lambda tokens: bm25_rank(tokens, INDEX, k1=3.0),
    "bm25-nolen": lambda tokens: bm25_rank(tokens, INDEX, b=0.0),
    "tfidf": tfidf_rank,
    "dense-12": _dense_small,
    "dense-24": _dense_mid,
    "dense-48": _dense_large,
    "hybrid": lambda tokens: reciprocal_rank_fusion(
        [bm25_rank(tokens, INDEX)[:50], _dense_mid(tokens)[:50]]),
}
RUNS = {name: {q.qid: rank(q.tokens)[:RUN_DEPTH] for q in QUERIES}
        for name, rank in SYSTEMS.items()}
LEXICAL = ("bm25", "bm25-k1-3", "bm25-nolen", "tfidf")     # systems that can only match words

print(f"{len(SYSTEMS)} systems; results returned per query (at most {RUN_DEPTH}):")
for _name, _run in RUNS.items():
    _sizes = [len(r) for r in _run.values()]
    print(f"  {_name:<11} mean {np.mean(_sizes):6.1f}  fewest {min(_sizes):3d}")

## 3. Exercise 1 — `recall_at_k()` and `reciprocal_rank()`

Both functions read a ranked list (best first) and `relevance`, a mapping from document id to
grade. A document is **relevant** when its grade is above zero; a document that is missing from
`relevance`, or listed with grade 0, is not. The same rule will matter a lot once judgements
are incomplete: **an unjudged document is treated as not relevant.**

* **recall@k** is the share of the relevant documents that appear in the top `k` of the
  ranking: how many you found, over how many there were to find.
* **reciprocal rank** is one over the rank of the first relevant document (rank 1 is the top),
  or zero if the ranking holds none. The mean over queries is **MRR**.

This lesson's convention for a query with no relevant document at all: both metrics are `0.0`.
That is a choice, not a law: it keeps every query in the mean, and later you will count how
often it happens, because a mean that silently includes zeros is a mean about something else.

<details><summary>💡 Hint 1 — what to think about</summary>

Decide first what the set of relevant documents is, and where it comes from: it is the
judgements, not the ranking. What should recall divide by? And when the ranking is shorter than
`k`, or `relevance` is empty, what must still not crash?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Recall: take the relevant ids from `relevance`, count how many of them occur among the first `k`
ranked ids, and divide by the number of relevant ids, returning zero when there are none.
Reciprocal rank: walk down the ranking from rank 1, stop at the first relevant id, and return
one over its position; if the walk ends without a hit, return zero.
</details>

In [ ]:
def recall_at_k(ranked: Sequence[int], relevance: Mapping[int, int], k: int) -> float:
    """The share of the relevant documents that appear in the top `k` of `ranked`.

    `ranked` lists document ids, best first, without repeats. `relevance` maps a document id to
    its non-negative integer grade; grade 0 and absence both mean "not relevant".

    Example:
        >>> recall_at_k([5, 1, 2], {1: 2, 9: 1, 4: 0}, k=2)
        0.5
        (document 1 is in the top 2; document 9 is not; document 4 is not relevant at all)

    Returns:
        a float in [0, 1]; `0.0` when `relevance` holds no relevant document.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def reciprocal_rank(ranked: Sequence[int], relevance: Mapping[int, int]) -> float:
    """One over the rank of the first relevant document in `ranked`, or 0.0 if there is none.

    Ranks start at 1. The whole ranking is searched, not only the top `k`.

    Example:
        >>> reciprocal_rank([5, 1, 2], {1: 1, 2: 2})
        0.5
        >>> reciprocal_rank([5, 6], {1: 1})
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_binary_metrics() -> None:
    cut = (recall_at_k([8, 6, 3], {3: 1}, 2), recall_at_k([8, 6, 3], {3: 1}, 3))
    assert cut == (0.0, 1.0), (
        f"got {cut}, expected (0.0, 1.0): the only relevant document is at rank 3, so it is "
        "outside the top 2 and inside the top 3; the top k is ranked[:k], not one more or one less"
    )
    got = recall_at_k([3, 1, 2, 9], {1: 1, 2: 1, 3: 1, 4: 1, 5: 1}, 4)
    assert got == 3 / 5, (
        f"got {got!r}, expected 0.6: three of the five relevant documents are in the top 4, so "
        "recall divides by the number of RELEVANT documents, not by k"
    )
    got = recall_at_k([5, 1, 2], {1: 2, 9: 1, 4: 0}, 2)
    assert got == 0.5, (
        f"got {got!r}, expected 0.5: recall divides by the number of RELEVANT documents in "
        "`relevance` (grade above 0), not by k, and not by the length of the ranking"
    )
    assert recall_at_k([4, 1], {1: 1, 4: 0}, 2) == 1.0, (
        "a document listed with grade 0 is not relevant, so it is neither found nor owed: "
        "only grades above zero count"
    )
    assert recall_at_k([1, 2, 3], {7: 1}, 3) == 0.0, "no relevant document in the top k is recall 0.0"
    assert recall_at_k([1], {1: 1, 2: 1, 3: 1, 4: 1}, 50) == 0.25, (
        "a ranking shorter than k is fine: count what is there and still divide by all the "
        "relevant documents"
    )
    assert recall_at_k([1, 2], {}, 2) == 0.0, (
        "with no relevant document the convention here is 0.0, not an error and not 1.0"
    )
    assert reciprocal_rank([5, 1, 2], {1: 1, 2: 2}) == 0.5, (
        "the first relevant document is at rank 2, so the answer is 1/2 whatever its grade; "
        "ranks start at 1, not 0"
    )
    assert reciprocal_rank([5, 6], {1: 1}) == 0.0, "no relevant document in the ranking is 0.0"
    assert reciprocal_rank([3, 4], {3: 0, 4: 2}) == 0.5, (
        "a document with grade 0 is not relevant: skip it and keep looking"
    )
    print("exercise 1 looks right")


_try("exercise 1", _check_binary_metrics)

## 4. Exercise 2 — `ndcg_at_k()`

Recall and reciprocal rank treat relevance as yes or no. **nDCG@k** uses the grades, and rewards
putting the better document higher. Walk down the top `k` ranks; a document of grade `g` at rank
`r` adds `g / log2(r + 1)` to the **discounted cumulative gain** (DCG). That is why a grade-2
document at rank 1 is worth more than the same document at rank 5. Then divide by the DCG of the
best possible ordering of the judged documents (the **ideal** DCG), so a perfect list scores 1.

Two conventions to know about. This lesson uses the grade itself as the gain, as T06-L01 does;
the textbook form in `claims.yaml` uses `2^grade - 1`, which changes the numbers (not the idea).
So never compare an nDCG from this notebook with one from a library without checking which gain
it uses. And the ideal is built from `relevance` alone, so **what has been judged decides the
denominator**: you will see in a moment why that matters.

<details><summary>💡 Hint 1 — what to think about</summary>

There are two sums with the same shape: one over the ranking you were given, one over the best
ordering of the grades. What does the best ordering look like, and does it stop at `k` too?
What happens to a ranked document that has no entry in `relevance`? And what if the ideal sum is
zero?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sum grade over log base two of rank plus one for the first `k` ranked documents, using zero for
a document with no grade. For the ideal, sort the positive grades from the largest down, keep the
first `k`, and sum the same expression over their positions. Divide, but return zero when the
ideal sum is zero.
</details>

In [ ]:
def ndcg_at_k(ranked: Sequence[int], relevance: Mapping[int, int], k: int) -> float:
    """nDCG@k with the grade as the gain and a log2(rank + 1) discount.

    DCG@k    = sum over the first k ranked documents of grade / log2(rank + 1), ranks from 1.
    ideal    = the same sum over the k largest positive grades in `relevance`, largest first.
    nDCG@k   = DCG@k / ideal, and 0.0 when the ideal is zero (no relevant document).

    A ranked document with no entry in `relevance` has grade 0. Grades are non-negative integers.

    Example:
        >>> ndcg_at_k([1, 2], {1: 2, 2: 1}, k=2)
        1.0
        >>> round(ndcg_at_k([2, 1], {1: 2, 2: 1}, k=2), 4)
        0.8597
        (DCG = 1/1 + 2/log2(3); ideal = 2/1 + 1/log2(3))
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_ndcg() -> None:
    assert ndcg_at_k([1, 2], {1: 2, 2: 1}, 2) == 1.0, (
        "the ideal order is the grades from largest to smallest; this ranking IS that order, "
        "so it scores exactly 1"
    )
    expected = (1 / math.log2(2) + 2 / math.log2(3)) / (2 / math.log2(2) + 1 / math.log2(3))
    got = ndcg_at_k([2, 1], {1: 2, 2: 1}, 2)
    assert abs(got - expected) < 1e-9, (
        f"got {got:.6f}, expected {expected:.6f}: the discount for rank r is log2(r + 1), the "
        "gain is the grade itself, and the ideal comes from the sorted grades, not from your ranking"
    )
    assert ndcg_at_k([8, 9], {1: 2}, 2) == 0.0, "nothing relevant retrieved is 0.0"
    assert ndcg_at_k([1], {}, 3) == 0.0, "with no relevant document the ideal is 0, and the answer 0.0"
    one_deep = ndcg_at_k([7, 1], {1: 1, 2: 1, 3: 1}, 2)
    expected = (1 / math.log2(3)) / (1 / math.log2(2) + 1 / math.log2(3))
    assert abs(one_deep - expected) < 1e-9, (
        f"got {one_deep:.6f}, expected {expected:.6f}: the ideal sum also stops at k, so only the "
        "k largest grades enter it"
    )
    print("exercise 2 looks right")


_try("exercise 2", _check_ndcg)

### The truth leaderboard

With the metrics written, score every system against the planted truth. `score_run` is given: it
averages a metric over queries and also counts the queries that had no relevant document in the
judgements it was given (zero here, by construction; the label sweep in section 8 counts them
again, under labels that can miss every relevant document of a query).

In [ ]:
def score_run(run: Mapping[int, Sequence[int]], qrels: Mapping[int, Mapping[int, int]],
              k: int = EVAL_K) -> dict[str, float]:
    """Mean recall@k, MRR and nDCG@k of one system over every query in `qrels`. Given.

    `empty` is how many queries had no relevant document in `qrels`: those score 0.0 and stay in
    the mean, so a number computed from thin judgements is the mean of a different quantity.
    """
    recalls, rrs, ndcgs, empty = [], [], [], 0
    for qid, relevance in qrels.items():
        ranked = run.get(qid, [])
        recalls.append(recall_at_k(ranked, relevance, k))
        rrs.append(reciprocal_rank(ranked, relevance))
        ndcgs.append(ndcg_at_k(ranked, relevance, k))
        empty += not any(g > 0 for g in relevance.values())
    return {"recall": float(np.mean(recalls)), "mrr": float(np.mean(rrs)),
            "ndcg": float(np.mean(ndcgs)), "empty": empty}


def leaderboard(scores: Mapping[str, float]) -> list[str]:
    """System names from best to worst; equal scores break by name. Given."""
    return sorted(scores, key=lambda name: (-scores[name], name))


def _truth_leaderboard() -> None:
    TRUTH_SCORES.update({name: score_run(run, TRUTH) for name, run in RUNS.items()})
    print(f"against the planted truth, k = {EVAL_K}")
    print(f"  {'system':<11} {'recall@k':>9} {'MRR':>7} {'nDCG@k':>8}")
    for name in leaderboard({n: s["ndcg"] for n, s in TRUTH_SCORES.items()}):
        s = TRUTH_SCORES[name]
        print(f"  {name:<11} {s['recall']:9.3f} {s['mrr']:7.3f} {s['ndcg']:8.3f}")
    low = [n for n in LEXICAL if TRUTH_SCORES[n]["recall"] < 1.0]
    best_lexical = max(TRUTH_SCORES[n]["recall"] for n in LEXICAL)
    print(f"\nthe word-matching systems reach recall@k of at most {best_lexical:.3f}: "
          f"{len(low)} of {len(LEXICAL)} miss relevant documents they cannot see by words.")


TRUTH_SCORES: dict[str, dict[str, float]] = {}
_try("truth leaderboard", _truth_leaderboard, needs=("exercise 1", "exercise 2"))

## 5. Exercise 3 — pooling

Judging every document for every query is out of the question on a real collection, so teams
judge a **pool**: for each query, the union of the top `depth` documents of each of a few
systems. Everything outside the pool is never looked at, and the evaluation then treats it as
not relevant. `claims.yaml` has the sources for how pooling is described and for the bias it
can build into a collection.

`pooled_qrels` takes the ranked lists of the contributing systems and the full judgements
(here, the planted truth, standing in for a human who judges whatever they are shown), and
returns what a judge would have produced: for every query, one entry for **every document in
the pool**, with its grade; a pooled document that is not relevant is listed with grade 0.
That explicit zero matters in section 8, where a labeller can get it wrong.

Why this lesson cares: a system that did not contribute to the pool is scored against
judgements that were chosen by other systems. Its relevant documents that none of them
retrieved are not in the pool, so they count for nothing.

<details><summary>💡 Hint 1 — what to think about</summary>

The result has one entry per query in `qrels`, even a query that no run mentions. A document
is in the pool if ANY contributing system has it among its first `depth` results for that
query. Where does the grade of a pooled document come from, and what is it if the document is
not in `qrels` at all?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Start from an empty mapping for every query in `qrels`. For each contributing system and each
of its queries, take the first `depth` ids of the ranking and, for each, record the grade that
`qrels` gives it for that query, using zero when it has none. A document found by several
systems is simply recorded once.
</details>

In [ ]:
def pooled_qrels(runs: Mapping[str, Mapping[int, Sequence[int]]],
                 qrels: Mapping[int, Mapping[int, int]], depth: int) -> dict[int, dict[int, int]]:
    """The judgements a judge would produce for the pool of `runs` at `depth`.

    `runs[system][qid]` is that system's ranked list of document ids for the query. The pool for
    a query is the union, over the systems, of the first `depth` ids of that list. Every pooled
    document gets the grade `qrels[qid]` gives it, or 0 if it has none. Queries of `runs` that
    are not in `qrels` are ignored; a query of `qrels` that no run mentions gets an empty pool.

    Example:
        >>> pooled_qrels({"a": {0: [5, 1, 7]}, "b": {0: [1, 2]}}, {0: {1: 2, 2: 1, 9: 1}}, depth=2)
        {0: {5: 0, 1: 2, 2: 1}}
        (a contributes 5 and 1, b contributes 1 and 2; document 9 was never retrieved, so never judged)

    Returns:
        {qid: {doc_id: grade}} with one key per query of `qrels`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_pooled_qrels() -> None:
    runs = {"a": {0: [5, 1, 7], 3: [2]}, "b": {0: [1, 2]}}
    truth = {0: {1: 2, 2: 1, 9: 1}, 3: {2: 1}, 4: {6: 1}}
    got = pooled_qrels(runs, truth, 2)
    assert set(got) == {0, 3, 4}, (
        f"got queries {sorted(got)}: there must be one entry per query in `qrels`, including "
        "query 4, which no run retrieved anything for"
    )
    assert got[0] == {5: 0, 1: 2, 2: 1}, (
        f"got {got[0]}, expected {{5: 0, 1: 2, 2: 1}}: take the FIRST `depth` ids of each run, "
        "keep a judged-not-relevant document as an explicit 0, and never judge a document no run "
        "retrieved (document 9)"
    )
    assert got[4] == {}, "a query nobody retrieved for has an empty pool, not a missing key"
    assert pooled_qrels(runs, truth, 1)[0] == {5: 0, 1: 2}, (
        "depth 1 takes only each run's first document: an off-by-one here changes the pool"
    )
    print("exercise 3 looks right")


_try("exercise 3", _check_pooled_qrels)

### One system's pool, and what it does to everyone else

Suppose a team has only BM25 and judges its top `POOL_DEPTH` results for every query. We can
now score every system twice: against the truth, and against that pool. `judged@k` is the share
of a system's own top `EVAL_K` documents that the pool judged at all: the number a careful
reader asks for before believing a leaderboard.

In [ ]:
def judged_share(run: Mapping[int, Sequence[int]], judged: Mapping[int, Mapping[int, int]],
                 k: int = EVAL_K) -> float:
    """Mean over queries of the share of the top `k` documents that have any judgement. Given."""
    shares = []
    for qid, ranked in run.items():
        top = ranked[:k]
        shares.append(sum(d in judged[qid] for d in top) / len(top) if top else 1.0)
    return float(np.mean(shares))


def _pool_table(members: Sequence[str], depth: int) -> dict[str, dict[str, float]]:
    pooled = pooled_qrels({m: RUNS[m] for m in members}, TRUTH, depth)
    scores = {name: score_run(run, pooled) for name, run in RUNS.items()}
    print(f"pool of {', '.join(members)} at depth {depth}: "
          f"{np.mean([len(p) for p in pooled.values()]):.1f} documents judged per query, "
          f"{np.mean([sum(g > 0 for g in p.values()) for p in pooled.values()]):.1f} of them "
          f"relevant (the truth has {np.mean([len(t) for t in TRUTH.values()]):.1f})")
    print(f"  {'system':<11} {'in pool':>7} {'judged@k':>9} {'MRR':>15} {'recall@k':>15} {'nDCG@k':>15}")
    for name in RUNS:
        t, p = TRUTH_SCORES[name], scores[name]
        print(f"  {name:<11} {'yes' if name in members else 'no':>7} "
              f"{judged_share(RUNS[name], pooled):9.3f} "
              f"{t['mrr']:7.3f}->{p['mrr']:6.3f} {t['recall']:7.3f}->{p['recall']:6.3f} "
              f"{t['ndcg']:7.3f}->{p['ndcg']:6.3f}")
    return scores


def _one_system_pool() -> None:
    scores = _pool_table(["bm25"], POOL_DEPTH)
    higher = [n for n in RUNS if scores[n]["mrr"] > TRUTH_SCORES[n]["mrr"] + 1e-12]
    print(f"\nsystems whose pooled MRR is above their true MRR: {len(higher)} of {len(RUNS)}")
    outside = [n for n in RUNS if n != "bm25"]
    for metric in ("mrr", "recall", "ndcg"):
        inside_move = scores["bm25"][metric] - TRUTH_SCORES["bm25"][metric]
        outside_moves = [scores[n][metric] - TRUTH_SCORES[n][metric] for n in outside]
        print(f"  {metric:<7} pool member moves {inside_move:+.3f}; the other systems move "
              f"between {min(outside_moves):+.3f} and {max(outside_moves):+.3f}")
    top_true = leaderboard({n: s["ndcg"] for n, s in TRUTH_SCORES.items()})[0]
    top_pool = leaderboard({n: s["ndcg"] for n, s in scores.items()})[0]
    print(f"best system by nDCG@k: {top_true} on the truth, {top_pool} on the pool")
    print(f"queries with no relevant judged document: {scores['bm25']['empty']} of {len(QUERIES)}")


_try("one-system pool", _one_system_pool, needs=("exercise 1", "exercise 2", "exercise 3"))

Read the table before reading on. Three things are worth finding in it yourself.

* **MRR**: it can only fall under a pool. A relevant document that was never judged becomes
  "not relevant", which can only push a system's first relevant hit down the ranking. How many
  systems have a pooled MRR above their true MRR is printed above.
* **recall and nDCG**: they can rise. Both divide by something built from the judgements (the
  number of relevant documents, the ideal DCG), and a thin pool shrinks that denominator. A
  system whose results look like the pool's gains from that, while a system whose relevant
  results sit outside the pool loses on the top of the fraction. Compare the move of the pool
  member with the range of moves of the others.
* **the leaderboard**: when the moves differ by system, the order of the systems can change, and
  the printed best system is the first place to look.

## 6. Exercise 4 — `kendall_tau()`: how far did the order move?

A table of scores is hard to compare by eye. What a leaderboard is for is the **order**, and
Kendall's tau is a plain way to say how much two orders agree. Take every pair of systems. If
both score lists put the same one ahead, the pair is **concordant** (+1); if they disagree, it is
**discordant** (-1); if either list scores the two systems exactly equal, the pair is **tied** and
counts 0 (scores a thousandth apart are ordered, not tied). Then

`tau = (sum over all pairs of that +1 / -1 / 0) / (n (n - 1) / 2)`

so identical orders give 1, exactly reversed orders give -1, and 0 means no overall agreement.
For fewer than two items there are no pairs; return `0.0`. (Other tau variants divide
differently when there are ties; this one divides by all the pairs, ties included.)

<details><summary>💡 Hint 1 — what to think about</summary>

Each pair is looked at once, not twice, and the sign of a difference is all that matters for it.
What is the sign of a difference of two equal numbers, and what does that make a tied pair
contribute to the sum?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over every pair of positions with the first below the second. For each, take the sign of the
difference in the first list and the sign of the difference in the second, and add their
product. Divide the total by the number of pairs. Raise an error if the lists differ in length.
</details>

In [ ]:
def kendall_tau(x: Sequence[float], y: Sequence[float]) -> float:
    """Kendall's tau between two score lists over the same items, ties counting 0.

    For every pair of positions i < j: +1 if x and y order the pair the same way, -1 if they
    order it oppositely, 0 if either scores the two items exactly equal. Divide the sum by n (n - 1) / 2.

    Example:
        >>> kendall_tau([3, 2, 1], [30, 20, 10])
        1.0
        >>> round(kendall_tau([1, 2, 3], [1, 3, 2]), 4)
        0.3333
        (three pairs: two agree, one disagrees)

    Returns:
        a float in [-1, 1]; `0.0` for fewer than two items. Raises ValueError if the lengths differ.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_kendall_tau() -> None:
    assert kendall_tau([3, 2, 1], [9, 5, 1]) == 1.0, "the same order is tau 1"
    assert kendall_tau([1, 2, 3, 4], [4, 3, 2, 1]) == -1.0, "the reversed order is tau -1"
    got = kendall_tau([1, 2, 3], [1, 3, 2])
    assert abs(got - 1 / 3) < 1e-12, (
        f"got {got!r}, expected one third: of the 3 pairs, 2 agree and 1 disagrees, "
        "and the denominator counts every pair"
    )
    got = kendall_tau([1, 1, 2], [1, 2, 3])
    assert abs(got - 2 / 3) < 1e-12, (
        f"got {got!r}, expected two thirds: a pair tied in either list adds 0 to the sum, but "
        "it still counts in the denominator"
    )
    assert kendall_tau([5], [5]) == 0.0, "fewer than two items has no pairs: return 0.0"
    try:
        kendall_tau([1, 2], [1, 2, 3])
    except ValueError:
        pass
    else:
        raise AssertionError("lists of different length cannot be compared: raise ValueError")
    print("exercise 4 looks right")


_try("exercise 4", _check_kendall_tau)

### How far does a pool move the leaderboard?

Now vary what the pool is made of and how deep it goes. For each setting the table prints: how many
of the planted relevant documents the pool judged at all (**coverage**), the tau between the true
order of the systems (by nDCG@k) and the order the pool gives, and the best system under each.
A setting where tau is 1 does not reorder anything; the further below 1, the more the pool has
rearranged the systems.

In [ ]:
def _pool_sweep() -> None:
    names = list(RUNS)
    truth_scores = [TRUTH_SCORES[n]["ndcg"] for n in names]
    best_true = leaderboard({n: TRUTH_SCORES[n]["ndcg"] for n in names})[0]
    settings = [("bm25 only", ("bm25",)), ("lexical systems", LEXICAL), ("every system", tuple(names))]
    relevant_total = sum(len(t) for t in TRUTH.values())
    print(f"best system on the truth: {best_true}")
    print(f"  {'pool made of':<16} {'depth':>5} {'judged/query':>12} {'coverage':>9} {'tau':>7}  best on pool")
    coverage: dict[tuple[str, int], float] = {}
    for label, members in settings:
        for depth in (5, 10, 20, 50, 100):
            pooled = pooled_qrels({m: RUNS[m] for m in members}, TRUTH, depth)
            scores = {n: score_run(run, pooled)["ndcg"] for n, run in RUNS.items()}
            found = sum(g > 0 for p in pooled.values() for g in p.values())
            coverage[(label, depth)] = found / relevant_total
            tau = kendall_tau(truth_scores, [scores[n] for n in names])
            print(f"  {label:<16} {depth:>5} {np.mean([len(p) for p in pooled.values()]):12.1f} "
                  f"{coverage[(label, depth)]:9.3f} {tau:7.3f}  {leaderboard(scores)[0]}")
    a, b = coverage[("lexical systems", 10)], coverage[("lexical systems", 100)]
    if a == b:
        print(f"\nfor the lexical pool, ten times the depth judges no new relevant document "
              f"(coverage stays {a:.3f}): a pool cannot reach what its systems never retrieve.")
    else:
        print(f"\nfor the lexical pool, depth 10 to depth 100 moves coverage from {a:.3f} to {b:.3f}.")


_try("pool sweep", _pool_sweep, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

Questions for the table, and the answers are all in it:

* Does a deeper pool from the same systems fix the coverage, or does it stop helping at some
  depth? Look at where coverage stops changing, and at how many documents each word-matching
  system returns at all (section 2).
* What does adding the dense systems to the pool do to coverage and tau, compared with deepening
  a pool of word-matching systems?
* The classic defence is a leave-one-out test: rebuild the pool without one contributing
  system and re-score that system. The next cell runs it on the word-matching systems.

In [ ]:
def _leave_one_out() -> None:
    full_pool = pooled_qrels({m: RUNS[m] for m in LEXICAL}, TRUTH, POOL_DEPTH)
    print(f"leave-one-out on the pool of the word-matching systems, depth {POOL_DEPTH}")
    print(f"  {'left out':<11} {'nDCG@k in pool':>15} {'nDCG@k without itself':>22} {'change':>8}")
    changes = []
    for member in LEXICAL:
        others = pooled_qrels({m: RUNS[m] for m in LEXICAL if m != member}, TRUTH, POOL_DEPTH)
        inside = score_run(RUNS[member], full_pool)["ndcg"]
        outside = score_run(RUNS[member], others)["ndcg"]
        changes.append(abs(outside - inside))
        print(f"  {member:<11} {inside:15.3f} {outside:22.3f} {outside - inside:+8.3f}")
    dense_gap = [score_run(RUNS[n], full_pool)["ndcg"] - TRUTH_SCORES[n]["ndcg"]
                 for n in RUNS if n not in LEXICAL]
    print(f"largest leave-one-out change: {max(changes):.3f}; the systems that never "
          f"contributed (the dense ones and the hybrid) are off by between {min(dense_gap):+.3f} and {max(dense_gap):+.3f} "
          "from their true nDCG@k.")


_try("leave-one-out", _leave_one_out,
     needs=("exercise 1", "exercise 2", "exercise 3"))

The leave-one-out test asks whether a system would score differently if the pool had never seen
it. When the pool members retrieve nearly the same documents, removing one barely changes the
pool, so the test reports little even where the pool is badly incomplete for systems that are
different in kind. Compare the two numbers on the last line printed above before deciding how
much comfort a small leave-one-out change gives.

## 7. Synthetic labels: a labeller whose mistakes you planted

Judging by hand is slow, so some teams let a model label the pooled pairs (`claims.yaml` cites a
2023 paper that reports doing this at scale). There is no model in this lesson, and there does
not need to be: what matters is the method. A labeller is a
function that takes a (query, document) pair and returns a grade, and it is wrong some of the
time. You can only know how wrong by comparing it with trusted labels on a sample, and here the
trusted labels are the planted truth.

The scripted labeller makes two kinds of mistake, at rates you choose:

* a **miss**: a truly relevant document is labelled 0. The rate may depend on the true grade
  (`miss_rates` maps a grade to its miss probability; a grade not in the mapping is never missed);
* a **false alarm**: a truly non-relevant document is labelled grade 1, at probability
  `false_alarm`.

Which pairs get it wrong must be reproducible and must not depend on the order you visit them,
so each pair gets its own fixed number `u` in [0, 1) from `_unit(seed, qid, doc)`, given below.
The mistake happens when `u` is **strictly below** the rate.

<details><summary>💡 Hint 1 — what to think about</summary>

Every judged pair has a true grade, and that grade decides which rate applies, and which kind of
mistake is possible. A rate of zero must never fire and a rate of one must always fire: which
comparison between `u` and the rate gives both? And the input mapping belongs to the caller.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build a new mapping with the same queries and the same documents. For each judged pair, get its
`u` once. If the true grade is above zero, look its rate up in `miss_rates` (zero when absent)
and either keep the grade or replace it with 0. If the true grade is zero, either keep 0 or
replace it with 1, using `false_alarm`.
</details>

In [ ]:
_MASK = (1 << 64) - 1


def _unit(seed: int, qid: int, doc: int) -> float:
    """A number in [0, 1) fixed by (seed, qid, doc): a 64-bit integer hash, the same on any
    machine and in any order. Given."""
    z = (seed * 0x9E3779B97F4A7C15 + qid * 0xBF58476D1CE4E5B9
         + doc * 0x94D049BB133111EB + 0x2545F4914F6CDD1D) & _MASK
    z ^= z >> 30
    z = (z * 0xBF58476D1CE4E5B9) & _MASK
    z ^= z >> 27
    z = (z * 0x94D049BB133111EB) & _MASK
    z ^= z >> 31
    return (z >> 11) / float(1 << 53)


def apply_labeller(judged: Mapping[int, Mapping[int, int]], seed: int,
                   miss_rates: Mapping[int, float], false_alarm: float) -> dict[int, dict[int, int]]:
    """Relabel every judged pair with a labeller that makes planted mistakes.

    For each `(qid, doc)` with true grade `g` in `judged`, let `u = _unit(seed, qid, doc)`:
      * `g > 0`: the new label is 0 if `u < miss_rates.get(g, 0.0)`, else `g`;
      * `g == 0`: the new label is 1 if `u < false_alarm`, else 0.

    Every query and every document of `judged` appears in the result; `judged` is not modified.

    Example:
        >>> apply_labeller({0: {4: 2, 5: 0}}, seed=1, miss_rates={}, false_alarm=0.0)
        {0: {4: 2, 5: 0}}
        (rates of zero never change a label)
        >>> apply_labeller({0: {4: 2, 5: 0}}, seed=1, miss_rates={2: 1.0}, false_alarm=1.0)
        {0: {4: 0, 5: 1}}
        (rates of one always do)

    Returns:
        {qid: {doc_id: label}}
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_apply_labeller() -> None:
    judged = {0: {4: 2, 5: 0, 6: 1}, 1: {4: 1, 9: 0}}
    snapshot = {q: dict(d) for q, d in judged.items()}
    assert apply_labeller(judged, 3, {}, 0.0) == snapshot, (
        "with every rate at zero no label may change: a mistake needs `u` strictly below the rate"
    )
    assert apply_labeller(judged, 3, {1: 1.0, 2: 1.0}, 1.0) == {0: {4: 0, 5: 1, 6: 0},
                                                                1: {4: 0, 9: 1}}, (
        "with every rate at one every relevant document is missed (label 0) and every "
        "non-relevant one is flagged (label 1)"
    )
    assert judged == snapshot, "the input must not be modified: build and return a new mapping"
    only_two = apply_labeller(judged, 3, {2: 1.0}, 0.0)
    assert only_two[0][6] == 1 and only_two[1][4] == 1 and only_two[0][4] == 0, (
        "the miss rate is looked up by the TRUE grade: here only grade-2 documents are missed"
    )
    u = _unit(3, 0, 4)
    just_below = apply_labeller({0: {4: 2}}, 3, {2: u}, 0.0)[0][4]
    just_above = apply_labeller({0: {4: 2}}, 3, {2: math.nextafter(u, 1.0)}, 0.0)[0][4]
    assert (just_below, just_above) == (2, 0), (
        "a mistake happens when u is strictly below the rate: at a rate equal to u the label "
        "must stay, one step above it the label must go"
    )
    print("exercise 5 looks right")


_try("exercise 5", _check_apply_labeller)

### Measure the labeller before you trust it

`label_error_rates` (given) compares labels with the judgements they came from and returns the
realised miss rate for each true grade and the realised false-alarm rate. Planted rates are
what you asked for; realised rates are what you got on a finite number of pairs. Run the labeller
over the pool that every system contributes to (the **fair pool**: every system's top
`POOL_DEPTH`), so each pair is judged.

In [ ]:
def label_error_rates(judged: Mapping[int, Mapping[int, int]],
                      labelled: Mapping[int, Mapping[int, int]]) -> dict:
    """Realised mistakes: {"miss": {grade: (count, rate)}, "false_alarm": (count, rate)}. Given."""
    seen: dict[int, list[int]] = {}
    alarm = [0, 0]
    for qid, docs in judged.items():
        for doc, grade in docs.items():
            wrong = labelled[qid][doc] != grade
            if grade > 0:
                row = seen.setdefault(grade, [0, 0])
                row[0] += 1
                row[1] += wrong
            else:
                alarm[0] += 1
                alarm[1] += wrong
    return {"miss": {g: (n, w / n) for g, (n, w) in sorted(seen.items())},
            "false_alarm": (alarm[0], alarm[1] / alarm[0] if alarm[0] else 0.0)}


FAIR_POOL: dict[int, dict[int, int]] = {}
FAIR_SCORES: dict[str, float] = {}


def _show_realised_rates() -> None:
    FAIR_POOL.update(pooled_qrels(RUNS, TRUTH, POOL_DEPTH))
    FAIR_SCORES.update({n: score_run(run, FAIR_POOL)["ndcg"] for n, run in RUNS.items()})
    names = list(RUNS)
    tau = kendall_tau([TRUTH_SCORES[n]["ndcg"] for n in names], [FAIR_SCORES[n] for n in names])
    print(f"fair pool: {sum(len(p) for p in FAIR_POOL.values())} judged pairs, "
          f"tau against the truth order {tau:.3f}")
    planted_miss, planted_alarm = {1: 0.30, 2: 0.10}, 0.20
    labelled = apply_labeller(FAIR_POOL, 0, planted_miss, planted_alarm)
    realised = label_error_rates(FAIR_POOL, labelled)
    rows = [(f"true grade {g}", n, "planted miss", planted_miss[g], r)
            for g, (n, r) in realised["miss"].items()]
    count, rate = realised["false_alarm"]
    rows.append(("true grade 0", count, "planted false alarm", planted_alarm, rate))
    near = 0
    for what, count, kind, planted, rate in rows:
        se = math.sqrt(planted * (1 - planted) / count)      # one standard error of a rate
        near += abs(rate - planted) <= 2 * se
        print(f"  {what}: {count:4d} pairs, {kind} {planted:.3f}, realised {rate:.3f} "
              f"(one standard error: {se:.3f})")
    print(f"{near} of {len(rows)} realised rates are within two standard errors of the planted "
          "rate, the size of difference that chance alone gives on this many pairs.")


_try("realised rates", _show_realised_rates, needs=("exercise 1", "exercise 2", "exercise 3",
                                                    "exercise 4", "exercise 5"))

## 8. How far does label error move the leaderboard?

Take the fair pool, score every system with clean labels, then relabel with each of the
labellers below at several error rates and score again. Each setting is repeated over `N_LABEL_SEEDS`
different seeds, because one unlucky draw proves nothing. The table prints, against the clean
order: the mean and the lowest tau, how many seeds changed which system is best, how far the
mean nDCG@k over all systems moved (the absolute scores, not the order), and how many queries per
seed were left with no relevant label at all (`empty`: they score 0.0 and stay in every mean).

* **forgetful**: misses relevant documents at one rate, whatever their grade;
* **trusting**: calls non-relevant documents relevant, and never misses;
* **keyword-minded**: misses documents with the lower true grade much more often than the
  higher one. In this collection those are the mismatch documents, which almost never share a
  word with the query (section 1 printed how many do). Whether a real labeller does this is for you to measure on your own data; here
  it is planted so you can see what such a mistake would do.

In [ ]:
N_LABEL_SEEDS = 20
LABELLERS = {
    "forgetful": [({1: m, 2: m}, 0.0) for m in (0.1, 0.3, 0.6, 0.85)],
    "trusting": [({}, f) for f in (0.1, 0.3, 0.6)],
    "keyword-minded": [({1: m, 2: 0.05}, 0.0) for m in (0.3, 0.6, 0.9)],
}
LABEL_TAU: dict[str, float] = {}       # setting -> mean tau over the seeds
LABEL_LOWEST: dict[str, float] = {}    # setting -> lowest tau of any one seed


def _describe(miss: Mapping[int, float], alarm: float) -> str:
    parts = [f"miss g{g} {r:.2f}" for g, r in sorted(miss.items())]
    if alarm:
        parts.append(f"false alarm {alarm:.2f}")
    return ", ".join(parts)


def _label_sweep() -> None:
    names = list(RUNS)
    clean = [FAIR_SCORES[n] for n in names]
    best_clean = leaderboard(dict(zip(names, clean)))[0]
    print(f"clean order on the fair pool: best system {best_clean}; "
          f"{N_LABEL_SEEDS} labelling seeds per row")
    print(f"  {'labeller':<15} {'planted rates':<24} {'mean tau':>9} {'lowest':>7} "
          f"{'best changed':>13} {'mean nDCG@k':>12} {'empty/seed':>11}")
    for labeller, settings in LABELLERS.items():
        for miss, alarm in settings:
            taus, changed, means, empties = [], 0, [], []
            for seed in range(N_LABEL_SEEDS):
                labelled = apply_labeller(FAIR_POOL, seed, miss, alarm)
                runs = [score_run(RUNS[n], labelled) for n in names]
                scores = [s["ndcg"] for s in runs]
                taus.append(kendall_tau(clean, scores))
                changed += leaderboard(dict(zip(names, scores)))[0] != best_clean
                means.append(np.mean(scores))
                empties.append(runs[0]["empty"])            # the same for every system
            LABEL_TAU[f"{labeller} {_describe(miss, alarm)}"] = float(np.mean(taus))
            LABEL_LOWEST[f"{labeller} {_describe(miss, alarm)}"] = min(taus)
            print(f"  {labeller:<15} {_describe(miss, alarm):<24} {np.mean(taus):9.3f} "
                  f"{min(taus):7.3f} {changed:>6d} of {N_LABEL_SEEDS:<3d}{np.mean(means):12.3f}"
                  f"{np.mean(empties):12.1f}")
    print(f"mean nDCG@k over the systems with clean labels: {np.mean(clean):.3f}; "
          f"queries with no relevant label under clean labels: "
          f"{score_run(RUNS[names[0]], FAIR_POOL)['empty']} of {len(FAIR_POOL)}")


_try("label sweep", _label_sweep, needs=("exercise 1", "exercise 2", "exercise 3",
                                         "exercise 4", "exercise 5"))

### Two kinds of damage

Label error can damage a leaderboard in two ways, and the table lets you separate them.

* **Absolute scores.** Look at the last column against the clean mean printed under the table.
  Does every labeller push the scores the same way? Where `empty/seed` is above zero, part of
  the fall is queries scored 0.0 because no relevant label was left, the averaging trap from
  exercise 1. An nDCG@k computed from one set of judgements must never be compared with one
  computed from another.
* **The order.** Compare the forgetful rows, where the same mistake lands alike on every
  system, with the keyword-minded rows, where the mistakes fall on the documents that only some
  systems retrieve. Which rows have the lowest tau, and which change the best system?

The next cell sets the worst row against the damage a pool does.

In [ ]:
def _compare_error_sources() -> None:
    if not LABEL_TAU:
        raise NotImplementedError("the label sweep above has not run")
    names = list(RUNS)
    truth = [TRUTH_SCORES[n]["ndcg"] for n in names]
    pooled = pooled_qrels({m: RUNS[m] for m in LEXICAL}, TRUTH, POOL_DEPTH)
    pool_tau = kendall_tau(truth, [score_run(RUNS[n], pooled)["ndcg"] for n in names])
    worst = min(LABEL_TAU, key=LABEL_TAU.get)
    unluckiest = min(LABEL_LOWEST, key=LABEL_LOWEST.get)
    print(f"pool of the word-matching systems at depth {POOL_DEPTH}: tau {pool_tau:.3f} against the truth")
    print(f"worst labeller setting in the sweep: mean tau {LABEL_TAU[worst]:.3f} over its "
          f"{N_LABEL_SEEDS} seeds  ({worst})")
    print(f"lowest tau of any single seed in the sweep: {LABEL_LOWEST[unluckiest]:.3f}  ({unluckiest})")
    if pool_tau < LABEL_TAU[worst]:
        print("on average, the pool moved the order further than the worst labeller did", end="")
        if LABEL_LOWEST[unluckiest] <= pool_tau:
            print("; one unlucky labelling seed moved it as far or further.")
        else:
            print(", and further than any single labelling seed.")
    else:
        print("on average, the worst labeller moved the order at least as far as the pool did.")
    print("both numbers belong to THIS collection and THESE systems; the lesson is to measure "
          "both on yours, not to carry either over.")


_try("pool versus labeller", _compare_error_sources,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 9. Common mistakes

* **Dividing recall by `k`.** Recall divides by how many relevant documents there are; dividing
  by `k` is precision, a different number that happens to look similar.
* **Building the ideal DCG from the ranking you were given.** Then every ranking scores 1 for
  its own ordering. The ideal comes from the judgements.
* **Forgetting that unjudged means not relevant.** That one convention is the whole pooling
  problem: it is invisible in the code and large in the numbers.
* **Reporting a leaderboard without `judged@k`.** A system whose top results are mostly
  unjudged is being scored on what other systems found.
* **Letting a pool be made only of systems that think alike.** A deeper pool of the same kind
  of system adds documents from the same place, and the leave-one-out test cannot see the gap.
* **Comparing nDCG across judgement sets.** The scores move with the labels; only the order is
  worth carrying over, and only after you have measured how much it moves.
* **Trusting a labeller you have not scored.** Its error rates are measurable on a small trusted
  sample, and the realised rates are noisy when the sample is small.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## 10. Self-check

1. A pool is built from one system's top results. A second system, which did not contribute to
   the pool, is scored against it. Its reciprocal rank on a query, compared with its reciprocal
   rank against the full truth, can:
   - (a) rise or fall, depending on the system
   - (b) stay the same or fall, because a relevant document that was never judged counts as not relevant
   - (c) not change at all, because ranks do not use judgements
   - (d) only rise, because the pool holds the easy documents

2. A system's nDCG@k is higher against a pool than against the full truth. Which explanation fits?
   - (a) Its relevant documents were judged twice
   - (b) The metric is broken, because a pooled score can never exceed the true one
   - (c) The pool contains relevant documents that the truth lacks
   - (d) The ideal DCG it is divided by was built from fewer judged relevant documents, and its
     own results sit mostly inside the pool

3. A labeller misses a fixed share of relevant documents, at random, whichever system retrieved
   them. Compared with what it does to the absolute scores, its effect on the ORDER of the
   systems is expected to be:
   - (a) smaller, because the same kind of mistake lands on every system, though close systems can still swap
   - (b) exactly as large
   - (c) nil: an order can never change when the mistakes are random
   - (d) larger, because random mistakes are the hardest kind to average out

4. System A scores 0.62 nDCG@10 on one set of judgements and 0.71 on another set built for the
   same collection. What can you conclude?
   - (a) A improved
   - (b) The second set of judgements is the more accurate one
   - (c) Nothing about A, until both numbers are computed against the same judgements
   - (d) A different metric must have been used

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

## What you built, and where it goes next

`recall_at_k`, `reciprocal_rank` and `ndcg_at_k` are the three numbers every later retrieval
lesson reports; `pooled_qrels` and `judged_share` are the two lines of defence for a number
someone else built the judgements for; `kendall_tau` and `apply_labeller` let you ask how much
a conclusion about which system is best depends on the labels it was drawn from. Hybrid fusion,
embedding fine-tuning and cross-lingual search all report nDCG, and each can now say, next to
it, how many of the documents behind it were judged.

Left out on purpose: significance tests between systems (T06-L03 has a paired bootstrap),
assessor disagreement between humans, and labellers whose error depends on the text. None can
be studied honestly until you can measure the simple case, which is what you just did.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_binary_metrics, ()),
        ("exercise 2", _check_ndcg, ()),
        ("exercise 3", _check_pooled_qrels, ()),
        ("exercise 4", _check_kendall_tau, ()),
        ("exercise 5", _check_apply_labeller, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.